In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Libraries loaded successfully')

### Let's load the Processed Data

In [ ]:
monthly = pd.read_csv('data/processed/monthly_aggregated.csv', parse_dates=['ds'])
monthly = monthly.sort_values('ds').reset_index(drop=True)

df = pd.read_csv('data/processed/cleaned_transactions.csv', parse_dates=['gl_date'])

print(f'Monthly data loaded   : {len(monthly)} months')
print(f'Transaction data loaded: {len(df):,} rows')
print()
monthly.head()

### Plot the Overall Time Series

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.suptitle('F42119 — Monthly Sales Trend (Jan 2023 to Dec 2025)', fontsize=15, fontweight='bold')

axes[0].plot(monthly['ds'], monthly['total_qty'], color='steelblue', linewidth=2.5, marker='o', markersize=5)
axes[0].fill_between(monthly['ds'], monthly['total_qty'], alpha=0.1, color='steelblue')
axes[0].set_title('Monthly Quantity Shipped (Units)', fontweight='bold')
axes[0].set_ylabel('Quantity (Units)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for year in [2024, 2025]:
    axes[0].axvline(pd.Timestamp(f'{year}-01-01'), color='red', linestyle='--', alpha=0.4, linewidth=1)
    axes[0].text(pd.Timestamp(f'{year}-01-15'), monthly['total_qty'].max()*0.88, str(year), color='red', fontsize=9)

axes[1].plot(monthly['ds'], monthly['total_revenue'], color='darkorange', linewidth=2.5, marker='o', markersize=5)
axes[1].fill_between(monthly['ds'], monthly['total_revenue'], alpha=0.1, color='darkorange')
axes[1].set_title('Monthly Revenue (LAK Billions)', fontweight='bold')
axes[1].set_ylabel('Revenue (LAK)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.1f}B'))

for year in [2024, 2025]:
    axes[1].axvline(pd.Timestamp(f'{year}-01-01'), color='red', linestyle='--', alpha=0.4, linewidth=1)
    axes[1].text(pd.Timestamp(f'{year}-01-15'), monthly['total_revenue'].max()*0.88, str(year), color='red', fontsize=9)

plt.tight_layout()
plt.savefig('reports/01_overall_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/01_overall_trend.png')

### Year Comparision
- Compare the same months across 2023, 2024 and 2025

In [ ]:
monthly['year']  = monthly['ds'].dt.year
monthly['month'] = monthly['ds'].dt.month

yoy_qty = monthly.pivot(index='month', columns='year', values='total_qty')
yoy_rev = monthly.pivot(index='month', columns='year', values='total_revenue')

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
colors = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Year-over-Year Comparison: 2023 vs 2024 vs 2025', fontsize=14, fontweight='bold')

for i, year in enumerate([2023, 2024, 2025]):
    if year in yoy_qty.columns:
        axes[0].plot(month_labels, yoy_qty[year], marker='o', linewidth=2, color=colors[i], label=str(year), markersize=6)
axes[0].set_title('Monthly Quantity YoY', fontweight='bold')
axes[0].set_ylabel('Quantity (Units)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].legend()

for i, year in enumerate([2023, 2024, 2025]):
    if year in yoy_rev.columns:
        axes[1].plot(month_labels, yoy_rev[year], marker='o', linewidth=2, color=colors[i], label=str(year), markersize=6)
axes[1].set_title('Monthly Revenue YoY (LAK)', fontweight='bold')
axes[1].set_ylabel('Revenue (LAK)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.0f}B'))
axes[1].legend()

plt.tight_layout()
plt.savefig('reports/02_yoy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
monthly['qty_mom_pct'] = monthly['total_qty'].pct_change() * 100
monthly['rev_mom_pct'] = monthly['total_revenue'].pct_change() * 100
monthly['qty_yoy_pct'] = monthly['total_qty'].pct_change(12) * 100
monthly['rev_yoy_pct'] = monthly['total_revenue'].pct_change(12) * 100

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
fig.suptitle('Growth Rates: Month-over-Month and Year-over-Year', fontsize=14, fontweight='bold')

colors_mom = ['green' if x >= 0 else 'red' for x in monthly['qty_mom_pct'].fillna(0)]
axes[0].bar(monthly['ds'], monthly['qty_mom_pct'], color=colors_mom, alpha=0.7, width=20)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Month-over-Month Quantity Growth (%)', fontweight='bold')
axes[0].set_ylabel('MoM Growth (%)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

yoy_data = monthly.dropna(subset=['qty_yoy_pct'])
colors_yoy = ['green' if x >= 0 else 'red' for x in yoy_data['qty_yoy_pct']]
axes[1].bar(yoy_data['ds'], yoy_data['qty_yoy_pct'], color=colors_yoy, alpha=0.7, width=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Year-over-Year Quantity Growth (%)', fontweight='bold')
axes[1].set_ylabel('YoY Growth (%)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.savefig('reports/03_growth_rates.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== AVERAGE GROWTH RATES ===')
print(f'Avg MoM Qty Growth : {monthly["qty_mom_pct"].mean():.1f}%')
print(f'Avg YoY Qty Growth : {monthly["qty_yoy_pct"].mean():.1f}%')
print(f'Avg MoM Rev Growth : {monthly["rev_mom_pct"].mean():.1f}%')
print(f'Avg YoY Rev Growth : {monthly["rev_yoy_pct"].mean():.1f}%')

### Quaterly Breakdown

In [ ]:
monthly['quarter'] = monthly['ds'].dt.quarter

quarterly = monthly.groupby(['year', 'quarter']).agg(
    total_qty     = ('total_qty', 'sum'),
    total_revenue = ('total_revenue', 'sum')
).reset_index()
quarterly['label'] = quarterly['year'].astype(str) + ' Q' + quarterly['quarter'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Quarterly Sales Breakdown by Year', fontsize=14, fontweight='bold')
palette = sns.color_palette('Set2', 4)

bars1 = axes[0].bar(quarterly['label'], quarterly['total_qty'],
                     color=[palette[q-1] for q in quarterly['quarter']], alpha=0.85)
axes[0].set_title('Quarterly Quantity (Units)', fontweight='bold')
axes[0].set_ylabel('Total Quantity')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].tick_params(axis='x', rotation=45)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                 f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=7)

bars2 = axes[1].bar(quarterly['label'], quarterly['total_revenue'],
                     color=[palette[q-1] for q in quarterly['quarter']], alpha=0.85)
axes[1].set_title('Quarterly Revenue (LAK Billions)', fontweight='bold')
axes[1].set_ylabel('Revenue (LAK)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.0f}B'))
axes[1].tick_params(axis='x', rotation=45)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                 f'{bar.get_height()/1e9:,.0f}B', ha='center', va='bottom', fontsize=7)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=palette[i], label=f'Q{i+1}') for i in range(4)]
axes[1].legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.savefig('reports/04_quarterly_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

### Plot the Top Items by Revenue

In [ ]:
item_summary = df.groupby('description').agg(
    total_qty     = ('qty_shipped', 'sum'),
    total_revenue = ('extended_price', 'sum'),
    order_count   = ('order_number', 'nunique')
).reset_index().sort_values('total_revenue', ascending=False)

item_summary['revenue_pct']    = (item_summary['total_revenue'] / item_summary['total_revenue'].sum() * 100).round(1)
item_summary['cumulative_pct'] = item_summary['revenue_pct'].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Item-Level Revenue Analysis (All 20 Items)', fontsize=14, fontweight='bold')

colors_items = sns.color_palette('Blues_r', len(item_summary))
bars = axes[0].barh(item_summary['description'], item_summary['total_revenue'],
                     color=colors_items, alpha=0.9)
axes[0].set_title('Total Revenue by Item (LAK)', fontweight='bold')
axes[0].set_xlabel('Revenue (LAK)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.0f}B'))
axes[0].invert_yaxis()
for bar, pct in zip(bars, item_summary['revenue_pct']):
    axes[0].text(bar.get_width()*1.01, bar.get_y() + bar.get_height()/2,
                 f'{pct}%', va='center', fontsize=8)

top5 = item_summary.head(5)
rest = item_summary.iloc[5:]['total_revenue'].sum()
pie_labels = list(top5['description']) + ['Others']
pie_values = list(top5['total_revenue']) + [rest]
axes[1].pie(pie_values, labels=pie_labels, autopct='%1.1f%%',
             startangle=140, colors=sns.color_palette('Set3', len(pie_labels)))
axes[1].set_title('Revenue Share: Top 5 Items vs Rest', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/05_item_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== TOP 10 ITEMS BY REVENUE ===')
print(item_summary[['description','total_qty','total_revenue','revenue_pct','cumulative_pct']].head(10).to_string(index=False))

### Correlation between Quantity & Revenue & Orders

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Correlation Analysis', fontsize=14, fontweight='bold')

axes[0].scatter(monthly['total_qty'], monthly['total_revenue'],
                color='steelblue', alpha=0.7, s=60, edgecolors='white')
z = np.polyfit(monthly['total_qty'], monthly['total_revenue'], 1)
p = np.poly1d(z)
axes[0].plot(sorted(monthly['total_qty']), p(sorted(monthly['total_qty'])),
             'r--', linewidth=1.5, label='Trend')
corr_qr = monthly['total_qty'].corr(monthly['total_revenue'])
axes[0].set_title(f'Quantity vs Revenue (r = {corr_qr:.3f})', fontweight='bold')
axes[0].set_xlabel('Quantity (Units)')
axes[0].set_ylabel('Revenue (LAK)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.0f}B'))
axes[0].legend()

corr_cols = monthly[['total_qty', 'total_revenue', 'order_count']].copy()
corr_cols.columns = ['Quantity', 'Revenue', 'Orders']
sns.heatmap(corr_cols.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, ax=axes[1], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/06_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Qty-Revenue correlation: {corr_qr:.3f}')

### Time Series Decomposition

In [ ]:
ts_qty = monthly.set_index('ds')['total_qty']
decomposition = seasonal_decompose(ts_qty, model='additive', period=12, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
fig.suptitle('Time Series Decomposition: Monthly Quantity Shipped', fontsize=14, fontweight='bold')

decomposition.observed.plot(ax=axes[0], color='steelblue', linewidth=2)
axes[0].set_ylabel('Observed')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

decomposition.trend.plot(ax=axes[1], color='darkorange', linewidth=2)
axes[1].set_ylabel('Trend')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

decomposition.seasonal.plot(ax=axes[2], color='green', linewidth=2)
axes[2].set_ylabel('Seasonality')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

decomposition.resid.plot(ax=axes[3], color='red', linewidth=1.5)
axes[3].set_ylabel('Residual')
axes[3].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/07_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

### Stationarity Test (ADF Test)

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f'=== ADF Test: {name} ===')
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'p-value       : {result[1]:.4f}')
    for key, val in result[4].items():
        print(f'   {key}: {val:.4f}')
    if result[1] < 0.05:
        print(f'STATIONARY — p-value {result[1]:.4f} < 0.05')
    else:
        print(f'NON-STATIONARY — p-value {result[1]:.4f} > 0.05 (Prophet handles this automatically)')
    print()

adf_test(monthly['total_qty'], 'Monthly Quantity')
adf_test(monthly['total_revenue'], 'Monthly Revenue')

### Ramp-Up Period Analysis

In [ ]:
ramp_up = monthly[monthly['ds'] < '2023-06-01']
normal  = monthly[monthly['ds'] >= '2023-06-01']

print('=== RAMP-UP PERIOD (Jan-May 2023) ===')
print(f'Avg Monthly Qty     : {ramp_up["total_qty"].mean():,.0f} units')
print(f'Avg Monthly Revenue : LAK {ramp_up["total_revenue"].mean():,.0f}')
print()
print('=== NORMAL OPERATIONS (Jun 2023 - Dec 2025) ===')
print(f'Avg Monthly Qty     : {normal["total_qty"].mean():,.0f} units')
print(f'Avg Monthly Revenue : LAK {normal["total_revenue"].mean():,.0f}')
print()
print(f'Ramp-up avg qty is {normal["total_qty"].mean() / ramp_up["total_qty"].mean():.1f}x lower than normal operations')
print('Prophet changepoint detection will automatically identify this structural break')

### EDA SUMMARY

In [ ]:
print('=' * 60)
print('   EDA SUMMARY: F42119 SALES ANALYSIS')
print('=' * 60)
print(f'Period          : Jan 2023 - Dec 2025 (36 months)')
print(f'Total Qty Sold  : {monthly["total_qty"].sum():,.0f} units')
print(f'Total Revenue   : LAK {monthly["total_revenue"].sum():,.0f}')
print(f'Total Orders    : {monthly["order_count"].sum():,.0f}')
print(f'Unique Items    : 20')
print()
print('KEY FINDINGS:')
avg_2025 = monthly[monthly['year']==2025]['total_qty'].mean()
avg_2024 = monthly[monthly['year']==2024]['total_qty'].mean()
print(f'  1. Strong upward trend: 2025 avg qty is {avg_2025/avg_2024:.1f}x higher than 2024')
print(f'  2. Ramp-up anomaly in Jan-May 2023: volumes ~{normal["total_qty"].mean()/ramp_up["total_qty"].mean():.0f}x lower than normal')
print(f'  3. Structural break at Jun 2023: business reached full operational capacity')
print(f'  4. Qty and Revenue highly correlated (r = {monthly["total_qty"].corr(monthly["total_revenue"]):.3f})')
print()
print('MODELLING IMPLICATIONS:')
print('  Prophet : use changepoint_prior_scale to capture Jun 2023 structural break')
print('  LSTM    : consider training only on Jun 2023 onwards for cleaner signal')
print('  Target  : forecast total quantity and revenue for Jan 2026')